# COLAB_MECHANISM_PROBES_GPT2

用于补做三类机制探针：

1. FT `round_8` 风险回弹是否对应更大的参数更新范数
2. ROME 为什么会出现 `ESR 低但 fairness risk 下降大`
3. MEMIT 为什么会出现 `ESR 高但公平性改善小`

建议运行环境：Colab GPU。


In [ ]:
# Purpose: Mount Google Drive to access project files.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Purpose: Initialize Colab and install dependencies.
%cd /content
!rm -rf /content/EasyEdit /content/work
!git clone --depth 1 https://github.com/zjunlp/EasyEdit.git

!pip -q install -U pip setuptools wheel
!pip -q install "PyYAML==6.0.3"
!pip -q install "datasets==2.20.0"
!pip -q install "transformers==4.57.1" "tokenizers==0.22.1" "sentence-transformers==3.2.1"
!pip -q install accelerate sentencepiece tqdm pandas numpy requests matplotlib hydra-core omegaconf einops higher iopath fairscale timm peft
!pip -q install "rouge==1.0.1" "av==14.2.0" qwen_vl_utils zhipuai "pyjwt==2.8.0"


In [ ]:
# Purpose: Copy scripts, local hparams, and data from your Drive project into /content/work.
import os, shutil, glob
from pathlib import Path

DST = '/content/work'
os.makedirs(DST, exist_ok=True)

candidates = glob.glob('/content/drive/MyDrive/**/scripts/run_edit_fairness_rounds.py', recursive=True)
if not candidates:
    raise FileNotFoundError('scripts/run_edit_fairness_rounds.py not found in MyDrive; please sync your project first')

src_script = candidates[0]
ROOT = os.path.dirname(os.path.dirname(src_script))
print('Detected ROOT =', ROOT)

required_scripts = [
    'scripts/run_edit_fairness_rounds.py',
    'scripts/run_fairness_eval.py',
    'scripts/probe_update_norm.py',
    'scripts/probe_samplewise_fairness.py',
]
for rel in required_scripts:
    src = os.path.join(ROOT, rel)
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    shutil.copy2(src, os.path.join(DST, os.path.basename(rel)))

hparams_dst = Path(DST) / 'hparams_custom'
hparams_dst.mkdir(parents=True, exist_ok=True)
for src in glob.glob(os.path.join(ROOT, 'hparams_custom', '*.yaml')):
    shutil.copy2(src, hparams_dst / os.path.basename(src))

required_data = [
    'data/edits_bias_stress_120.json',
    'data/crows_pairs_sample.jsonl',
    'data/bbq_sample.jsonl',
]
for rel in required_data:
    src = os.path.join(ROOT, rel)
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    shutil.copy2(src, os.path.join(DST, os.path.basename(src)))

print('copied files in /content/work:', sorted(os.listdir(DST)))
print('copied hparams:', sorted(os.listdir(hparams_dst)))


In [ ]:
# Purpose: Patch EasyEdit imports and compatibility issues for stable Colab runs.
from pathlib import Path

nethook_py = Path('/content/EasyEdit/easyeditor/util/nethook.py')
nt = nethook_py.read_text(encoding='utf-8')
old_sig = "def retain_hook(m, inputs, output, kwargs=None):"
new_sig = "def retain_hook(m, inputs, kwargs, output):"
if old_sig in nt:
    nt = nt.replace(old_sig, new_sig)
old_legacy = """            def legacy_hook(m, inputs, output):
                return retain_hook(m, inputs, output, kwargs=None)
"""
new_legacy = """            def legacy_hook(m, inputs, output):
                return retain_hook(m, inputs, None, output)
"""
if old_legacy in nt:
    nt = nt.replace(old_legacy, new_legacy)
nethook_py.write_text(nt, encoding='utf-8')
print('patched nethook signature')

init_py = Path('/content/EasyEdit/easyeditor/__init__.py')
init_py.write_text(
    "from .editors.editor import BaseEditor\n"
    "from .models.ft.ft_hparams import FTHyperParams\n"
    "from .models.rome.rome_hparams import ROMEHyperParams\n"
    "from .models.memit.memit_hparams import MEMITHyperParams\n"
    "from .models.mend.mend_hparams import MENDHyperParams\n"
    "from .models.grace.grace_hparams import GraceHyperParams\n"
    "from .models.wise.wise_hparams import WISEHyperParams\n",
    encoding='utf-8'
)

alg_dict_py = Path('/content/EasyEdit/easyeditor/util/alg_dict.py')
alg_dict_py.write_text(
    "from ..models.rome import apply_rome_to_model\n"
    "from ..models.memit import apply_memit_to_model\n"
    "from ..models.mend import MendRewriteExecutor\n"
    "from ..models.ft import apply_ft_to_model\n"
    "from ..models.grace import apply_grace_to_model\n"
    "from ..models.wise import apply_wise_to_model\n\n"
    "ALG_DICT = {\n"
    "    'ROME': apply_rome_to_model,\n"
    "    'MEMIT': apply_memit_to_model,\n"
    "    'MEND': MendRewriteExecutor().apply_to_model,\n"
    "    'FT': apply_ft_to_model,\n"
    "    'GRACE': apply_grace_to_model,\n"
    "    'WISE': apply_wise_to_model,\n"
    "}\n\n"
    "ALG_MULTIMODAL_DICT = {}\n"
    "PER_ALG_DICT = {}\n"
    "DS_DICT = {}\n"
    "MULTIMODAL_DS_DICT = {}\n"
    "PER_DS_DICT = {}\n"
    "Safety_DS_DICT = {}\n",
    encoding='utf-8'
)

models_init = Path('/content/EasyEdit/easyeditor/models/__init__.py')
models_init.write_text(
    "from .ft import *\n"
    "from .rome import *\n"
    "from .memit import *\n"
    "from .mend import *\n"
    "from .grace import *\n"
    "from .wise import *\n",
    encoding='utf-8'
)

editor_py = Path('/content/EasyEdit/easyeditor/editors/editor.py')
txt = editor_py.read_text(encoding='utf-8')
txt = txt.replace('from ..models.melo.melo import LORA', 'LORA = None  # patched: disable melo import')
txt = txt.replace('if isinstance(edited_model, LORA):', 'if (LORA is not None) and isinstance(edited_model, LORA):')
editor_py.write_text(txt, encoding='utf-8')
print('patched imports and editor guard')

compute_z_py = Path('/content/EasyEdit/easyeditor/models/memit/compute_z.py')
cz = compute_z_py.read_text(encoding='utf-8')
old = """        loss_layer_out = tr[hparams.layer_module_tmp.format(loss_layer)].output
        if isinstance(loss_layer_out, (list, tuple)):
            output = loss_layer_out[0]
        else:
            output = loss_layer_out
"""
new = """        loss_layer_out = tr[hparams.layer_module_tmp.format(loss_layer)].output
        if isinstance(loss_layer_out, (list, tuple)):
            output = loss_layer_out[0]
        elif isinstance(loss_layer_out, dict):
            output = None
            for k in ('hidden_states', 'last_hidden_state', 'output'):
                v = loss_layer_out.get(k, None)
                if isinstance(v, torch.Tensor):
                    output = v
                    break
                if isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], torch.Tensor):
                    output = v[0]
                    break
            if output is None:
                for _, v in loss_layer_out.items():
                    if isinstance(v, torch.Tensor):
                        output = v
                        break
            if output is None:
                raise TypeError(f'Unsupported dict output at loss layer: {list(loss_layer_out.keys())[:10]}')
        else:
            output = loss_layer_out
"""
if old in cz:
    cz = cz.replace(old, new)
    compute_z_py.write_text(cz, encoding='utf-8')
    print('patched MEMIT compute_z dict-output')
else:
    print('MEMIT compute_z patch skipped (pattern not found)')


In [ ]:
# Purpose: Build FT and MEMIT probe configs for GPT-2; reuse local ROME config if present.
import yaml
from pathlib import Path

ft_src = Path('/content/EasyEdit/hparams/FT/gpt2-xl.yaml')
ft_dst = Path('/content/work/FT_gpt2_probe.yaml')
ft_cfg = yaml.safe_load(ft_src.read_text(encoding='utf-8'))
ft_cfg['alg_name'] = 'FT'
ft_cfg['model_name'] = 'gpt2'
ft_cfg['device'] = 0
ft_cfg['batch_size'] = 1
ft_cfg['num_steps'] = 25
ft_cfg['model_parallel'] = False
ft_dst.write_text(yaml.safe_dump(ft_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('written:', ft_dst)

memit_src = Path('/content/EasyEdit/hparams/MEMIT/gpt2-xl.yaml')
memit_dst = Path('/content/work/MEMIT_gpt2_probe.yaml')
memit_cfg = yaml.safe_load(memit_src.read_text(encoding='utf-8'))
memit_cfg['alg_name'] = 'MEMIT'
memit_cfg['model_name'] = 'gpt2'
memit_cfg['device'] = 0
memit_cfg['layers'] = [5, 6, 7, 8, 9]
memit_cfg['v_loss_layer'] = 11
memit_cfg['mom2_dataset'] = 'wikitext'
memit_cfg['mom2_n_samples'] = 300
memit_cfg['mom2_dtype'] = 'float32'
memit_cfg['v_weight_decay'] = 0.5
memit_cfg['stats_dir'] = '/content/work/cache/stats'
memit_cfg['model_parallel'] = False
Path('/content/work/cache/stats').mkdir(parents=True, exist_ok=True)
memit_dst.write_text(yaml.safe_dump(memit_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('written:', memit_dst)

rome_dst = Path('/content/work/ROME_gpt2_probe.yaml')
local_rome = Path('/content/work/hparams_custom/ROME_gpt2_local.yaml')
if local_rome.exists():
    rome_dst.write_text(local_rome.read_text(encoding='utf-8'), encoding='utf-8')
    print('copied:', local_rome, '->', rome_dst)
else:
    rome_cfg = {
        'alg_name': 'ROME',
        'model_name': 'gpt2',
        'stats_dir': '/content/EasyEdit/data/stats',
        'device': 0,
        'layers': [9, 10, 11],
        'fact_token': 'subject_last',
        'v_num_grad_steps': 20,
        'v_lr': 5e-1,
        'v_loss_layer': 11,
        'v_weight_decay': 0.5,
        'clamp_norm_factor': 4,
        'kl_factor': 0.0625,
        'mom2_adjustment': False,
        'context_template_length_params': [[5, 10], [10, 10]],
        'rewrite_module_tmp': 'transformer.h.{}.mlp.c_proj',
        'layer_module_tmp': 'transformer.h.{}',
        'mlp_module_tmp': 'transformer.h.{}.mlp',
        'attn_module_tmp': 'transformer.h.{}.attn',
        'ln_f_module': 'transformer.ln_f',
        'lm_head_module': 'transformer.wte',
        'mom2_dataset': 'wikitext',
        'mom2_n_samples': 300,
        'mom2_dtype': 'float32',
        'model_parallel': False,
        'fp16': False,
    }
    rome_dst.write_text(yaml.safe_dump(rome_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('written synthesized ROME config:', rome_dst)


In [ ]:
# Purpose: Define probe paths.
EDITS_JSON = '/content/work/edits_bias_stress_120.json'
CROWS_JSONL = '/content/work/crows_pairs_sample.jsonl'
BBQ_JSONL = '/content/work/bbq_sample.jsonl'

FT_HPARAMS = '/content/work/FT_gpt2_probe.yaml'
ROME_HPARAMS = '/content/work/ROME_gpt2_probe.yaml'
MEMIT_HPARAMS = '/content/work/MEMIT_gpt2_probe.yaml'

print('EDITS_JSON =', EDITS_JSON)
print('CROWS_JSONL =', CROWS_JSONL)
print('BBQ_JSONL =', BBQ_JSONL)
print('FT_HPARAMS =', FT_HPARAMS)
print('ROME_HPARAMS =', ROME_HPARAMS)
print('MEMIT_HPARAMS =', MEMIT_HPARAMS)


## FT probe

目标：验证 FT `round_8` 风险回弹是否与更大的参数更新范数相关。


In [ ]:
!python /content/work/probe_update_norm.py   --hparams "$FT_HPARAMS"   --edits_json "$EDITS_JSON"   --rounds 7 8 9   --step 12   --out "/content/work/results/probe/ft_round789_update_norm.json"


In [ ]:
import json
from pprint import pprint

with open('/content/work/results/probe/ft_round789_update_norm.json', 'r', encoding='utf-8') as f:
    ft_norm = json.load(f)

pprint(ft_norm)


## ROME probe

目标：验证 ROME 为什么会出现 `ESR 低但 fairness risk 下降大`，先看样本级公平性分数变化。


In [ ]:
!python /content/work/probe_samplewise_fairness.py   --hparams "$ROME_HPARAMS"   --edits_json "$EDITS_JSON"   --crows "$CROWS_JSONL"   --bbq "$BBQ_JSONL"   --rounds 0 1 10   --step 12   --crows_limit 300   --bbq_limit 300   --out_dir "/content/work/results/probe/rome_samplewise"


In [ ]:
import json
from pprint import pprint

for r in [0, 1, 10]:
    path = f'/content/work/results/probe/rome_samplewise/round_{r}_samplewise.json'
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n===== ROME round {r} =====')
    pprint(data['crows_summary'])
    pprint(data['bbq_summary'])


## MEMIT probe

目标：验证 MEMIT 为什么会出现 `ESR 高但公平性改善小`，看样本级输出分布变化幅度是否更小。


In [ ]:
!python /content/work/probe_samplewise_fairness.py   --hparams "$MEMIT_HPARAMS"   --edits_json "$EDITS_JSON"   --crows "$CROWS_JSONL"   --bbq "$BBQ_JSONL"   --rounds 0 10   --step 12   --crows_limit 300   --bbq_limit 300   --out_dir "/content/work/results/probe/memit_samplewise"


In [ ]:
import json
from pprint import pprint

for r in [0, 10]:
    path = f'/content/work/results/probe/memit_samplewise/round_{r}_samplewise.json'
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n===== MEMIT round {r} =====')
    pprint(data['crows_summary'])
    pprint(data['bbq_summary'])


## Quick comparison

这一节用来快速比较 ROME 和 MEMIT 在样本级公平性上的总体变化方向。


In [ ]:
import json

def load_summary(path):
    with open(path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return {
        'round': d['round'],
        'crows_rate': d['crows_summary'].get('prefer_stereo_rate'),
        'bbq_acc': d['bbq_summary'].get('accuracy_proxy'),
        'mean_ppl_gap': d['crows_summary'].get('mean_ppl_gap'),
    }

rome0 = load_summary('/content/work/results/probe/rome_samplewise/round_0_samplewise.json')
rome10 = load_summary('/content/work/results/probe/rome_samplewise/round_10_samplewise.json')
memit0 = load_summary('/content/work/results/probe/memit_samplewise/round_0_samplewise.json')
memit10 = load_summary('/content/work/results/probe/memit_samplewise/round_10_samplewise.json')

print('ROME baseline -> round10')
print(rome0)
print(rome10)

print('\nMEMIT baseline -> round10')
print(memit0)
print(memit10)


In [ ]:
# Optional: inspect which CrowS samples changed the most for ROME.
import json
import pandas as pd

def load_crows_detail(path):
    with open(path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return pd.DataFrame(d['crows_detail'])

base_df = load_crows_detail('/content/work/results/probe/rome_samplewise/round_0_samplewise.json')
edit_df = load_crows_detail('/content/work/results/probe/rome_samplewise/round_10_samplewise.json')

merged = base_df.merge(edit_df, on='sample_id', suffixes=('_base', '_edit'))
merged['delta_p_more'] = merged['p_more_edit'] - merged['p_more_base']
merged['delta_p_less'] = merged['p_less_edit'] - merged['p_less_base']
merged['delta_gap'] = merged['ppl_gap_edit'] - merged['ppl_gap_base']

merged = merged.reindex(merged['delta_gap'].abs().sort_values(ascending=False).index)
merged.head(20)


In [ ]:
# Optional: inspect which BBQ samples changed prediction for ROME.
import json
import pandas as pd

def load_bbq_detail(path):
    with open(path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return pd.DataFrame(d['bbq_detail'])

base_df = load_bbq_detail('/content/work/results/probe/rome_samplewise/round_0_samplewise.json')
edit_df = load_bbq_detail('/content/work/results/probe/rome_samplewise/round_10_samplewise.json')

merged = base_df.merge(edit_df, on='sample_id', suffixes=('_base', '_edit'))
merged['pred_changed'] = merged['pred_base'] != merged['pred_edit']
merged[['sample_id', 'category_base', 'label_base', 'pred_base', 'pred_edit', 'pred_changed']].head(30)


In [ ]:
# Purpose: Zip outputs and download to local machine.
import os, subprocess
from google.colab import files

OUT_ROOT = '/content/work/results/probe'
zip_name = 'mechanism_probe_outputs.zip'
subprocess.run(['bash', '-lc', f"cd '{OUT_ROOT}' && zip -r '{zip_name}' ."], check=True)
zip_path = os.path.join(OUT_ROOT, zip_name)
print('zip =>', zip_path)
files.download(zip_path)
